In [1]:
import pandas as pd

In [2]:
d = {}
for i in range(9):
    d[i] = pd.read_json(f'../data/impactdir_extraction_results_{i}.jsonl', lines=True)

In [3]:
df = pd.concat(d.values(), ignore_index=True).set_index(['openalex_id', 'chunk_idx']).sort_index().reset_index()
df

,openalex_id,chunk_idx,cluster_id,impact_dim,impact_direction
0,W1000416519,0,0,Natural Resources - Forests,unknown
1,W1000416519,0,0,Natural Resources - Freshwater,unknown
2,W1000416519,0,0,Natural Resources - Urban land,unknown
3,W1000416519,0,0,Natural Resources - Wetlands,unknown
4,W1000416519,0,0,Wellbeing - Community,unknown
...,...,...,...,...,...
10086553,W991832166,5,206,Wellbeing - Community,unknown
10086554,W991832166,5,206,Human Needs - Education,unknown
10086555,W991832166,5,206,Justice - Distributional,unknown
10086556,W991832166,5,206,Justice - Procedural,unknown


In [18]:
df.to_parquet('../results_557k/impact_direction_raw_2026-03-12.parquet')

In [4]:
df.cluster_id.value_counts()

cluster_id
21      18821
187     17647
1049    17568
818     17470
1518    16759
        ...  
3034        9
3645        7
3037        4
3708        4
2047        2
Name: count, Length: 2625, dtype: int64

In [5]:
df['chunk_id'] = df['openalex_id'] + '_' + df['chunk_idx'].astype(str)

In [7]:
df['chunk_id'].nunique()

347164

In [8]:
df['openalex_id'].nunique()

226174

In [9]:
df.impact_direction.value_counts()

impact_direction
unknown     8921563
positive    1106267
neutral       50167
negative       8561
Name: count, dtype: int64

In [10]:
df = df[df['impact_direction'] != 'unknown']

In [11]:
df[['impact_category', 'impact_dim']] = df['impact_dim'].str.split(' - ', expand=True)

In [12]:
counts = df.groupby(['cluster_id', 'impact_category', 'impact_dim', 'impact_direction']).size().unstack(fill_value=0).reset_index()
counts = counts.rename_axis(None, axis=1)
refs = df.groupby(['cluster_id', 'impact_category', 'impact_dim', 'impact_direction']).apply(lambda x: list(x['chunk_id'])).unstack(fill_value=[]).reset_index()
refs = refs.rename_axis(None, axis=1)
refs = refs.rename(columns={'positive': 'positive_refs', 'negative': 'negative_refs', 'neutral': 'neutral_refs'})

counts = counts.merge(refs, on=['cluster_id', 'impact_category', 'impact_dim'], how='left')
counts

,cluster_id,impact_category,impact_dim,negative,neutral,positive,negative_refs,neutral_refs,positive_refs
0,0,Human Needs,Shelter and living conditions,0,0,13,[],[],"[W2003373721_1, W2156345725_0, W2289768102_1, ..."
1,0,Justice,Distributional,0,2,0,[],"[W2557096622_1, W4389432677_0]",[]
2,0,Planetary Boundaries,Climate change,0,0,1,[],[],[W4415811729_2]
3,0,Wellbeing,Community,0,0,1,[],[],[W4287095827_3]
4,0,Wellbeing,Environment,0,0,8,[],[],"[W1000416519_1, W2156345725_0, W2557096622_1, ..."
...,...,...,...,...,...,...,...,...,...
33683,3708,Wellbeing,Environment,0,0,1,[],[],[W4413291661_0]
33684,3759,Human Needs,Healthcare,0,0,1,[],[],[W4415006218_0]
33685,3759,Wellbeing,Community,0,0,2,[],[],"[W4415006218_0, W4415018649_1]"
33686,3759,Wellbeing,Jobs,0,0,1,[],[],[W4415018649_1]


In [13]:
counts.to_parquet('../results_557k/cluster_impacts_2026-03-12.parquet')

In [14]:
cat = counts.groupby(['cluster_id', 'impact_category'])[['negative', 'neutral', 'positive']].sum().reset_index()
cat

,cluster_id,impact_category,negative,neutral,positive
0,0,Human Needs,0,0,13
1,0,Justice,0,2,0
2,0,Planetary Boundaries,0,0,1
3,0,Wellbeing,0,0,30
4,1,Human Needs,0,0,3
...,...,...,...,...,...
9230,3702,Planetary Boundaries,0,0,2
9231,3702,Wellbeing,0,0,2
9232,3708,Wellbeing,0,0,1
9233,3759,Human Needs,0,0,1


In [15]:
def classify_cluster(group):
    g = group.set_index('impact_category')

    def get(category, direction):
        if category in g.index:
            return g.loc[category, direction]
        return 0

    N = 2  # minimum positive impacts threshold

    env_categories = ['Natural Resources', 'Planetary Boundaries']
    social_categories = ['Wellbeing', 'Justice', 'Human Needs']

    env_positive = sum(get(cat, 'positive') for cat in env_categories)
    env_neutral = sum(get(cat, 'neutral') for cat in env_categories)
    env_negative = sum(get(cat, 'negative') for cat in env_categories)
    
    social_positive = sum(get(cat, 'positive') for cat in social_categories)
    social_neutral = sum(get(cat, 'neutral') for cat in social_categories)
    social_negative = sum(get(cat, 'negative') for cat in social_categories)
    

    if env_positive > N and social_negative == 0:
        return 'S'   # Sufficiency
    elif env_negative > 0 or social_negative > 0:
        return 'NS'  # Not sufficiency
    elif env_positive == 0:
        return 'NR' # not relevant
    else:
        return 'PS'  # Potential sufficiency

cluster_classification = (
    cat.groupby('cluster_id')
    .apply(classify_cluster)
    .reset_index()
    .rename(columns={0: 'sustainability_class'})
)

cluster_classification['sustainability_class'].value_counts()

sustainability_class
S     1064
NR     683
NS     537
PS     197
Name: count, dtype: int64

In [ ]:
cluster_classification['sustainability_class'].value_counts()

sustainability_class
S     1064
NR     683
NS     537
PS     197
Name: count, dtype: int64

In [20]:
clusters = pd.read_csv('../clustering/cluster_representatives_2026-03-10.csv')

In [25]:
cc = clusters.merge(cluster_classification, on='cluster_id', how='left')
cc

,cluster_id,text,count,sustainability_class
0,0,Insurance programs and tax incentives for floo...,715,PS
1,1,Preserving natural lands for flood management,447,S
2,2,Continuous monitoring of fishways to optimise ...,1536,S
3,3,Increasing restoration funding,79,S
4,4,Implementation of measures to mitigate the ris...,1031,NS
...,...,...,...,...
2620,3673,Normativas de gestión de riesgos hídricos para...,3,NaN
2621,3702,Investments in infrastructure directly address...,4,PS
2622,3708,Target Environment-Specific Deterioration Mana...,6,NR
2623,3710,Equity-led policies,3,NaN


In [67]:
cc.to_csv('clusters_with_sustainability_classification_temp_2026-03-13.csv', index=False)

In [27]:
for cls, group in cc.groupby('sustainability_class'):
    print(f"Class {cls}:")
    print(group['text'].values.tolist())

Class NR:
['Collection of social and economic data to enhance vulnerability assessments', 'Establishment of online learning platforms in remote areas', 'Analysis of consumer perceptions and purchasing choices', 'Incentives to improve physical education teaching', 'Supporting coaches and peer leaders in the promotion of athletes’ well-being and reducing ill-being', 'Consider the impact of family structure, birth order, personality, and socioeconomic status in future research to obtain better results.', 'Appropriate sport and PA offerings for disabled persons', 'Future studies can be expanded to multiple regions and a larger sample size.', 'Implementing meat inspection in slaughterhouses and meat selling centers.', 'Support labour-intensive development programs to stimulate capital formation', 'Update tsunami evacuation plans to reflect changes in tsunami scenarios and arrival times', 'WHO Action Level for radon concentrations', 'Increase media exposure of health related knowledge', 'Tai

Clearly imperfect but we'll go with that for the moment, for integration.

## Eval
To finish.

In [29]:
gold = pd.read_parquet('../gold_impact_direction.parquet')
gold.head()

,openalex_id,chunk_idx,text,policy,impact_category,impact,direction
0,W1490165506,60,approximately 30% of the young homeless popul...,[SOCIAL] [PARTICIPATORY] [NATIONAL] Introduce ...,Justice,distributional,positive
1,W1490165506,60,approximately 30% of the young homeless popul...,[SOCIAL] [SPATIAL] [NATIONAL] Provide exit opt...,Justice,distributional,positive
2,W153963444,2,"former experiences, is an important constitue...",[NATURE] [ECONOMIC] [REGIONAL] Prize-money for...,PlanetaryBoundary,biosphere_integrity,positive
3,W153963444,2,"former experiences, is an important constitue...",[NATURE] [ECONOMIC] [REGIONAL] Prize-money for...,PlanetaryBoundary,climate_change,unknown
4,W153963444,2,"former experiences, is an important constitue...",[NATURE] [ECONOMIC] [REGIONAL] Prize-money for...,Wellbeing,community,unknown


In [31]:
gold['policy'] = gold['policy'].apply(lambda x: x.split(']')[-1].strip())

In [28]:
policies = pd.read_parquet('../results_557k/clustered_policies_with_representatives_2026-03-10.parquet')
policies.head()

,openalex_id,chunk_idx,text,cluster_id,representative
0,W1000416519,0,Buyouts of property and structures as part of ...,0,False
1,W1000416519,0,Prioritization of parcels for buyouts to reduc...,0,False
2,W1000416519,0,"Restoration of natural resources, including we...",1,False
3,W1000416519,0,Restoration of salmonid habitat in Sonoma Coun...,2,False
4,W1000416519,1,National Oceanic and Atmospheric Administratio...,3,False


In [64]:
gp = policies.merge(gold[['openalex_id', 'chunk_idx']], on=['openalex_id', 'chunk_idx'], how='inner')
gp

,openalex_id,chunk_idx,text,cluster_id,representative
0,W1490165506,60,Provide exit options for young people to leave...,540,False
1,W1490165506,60,Provide exit options for young people to leave...,540,False
2,W1490165506,60,Introduce youth quotas in parliaments to ensur...,630,False
3,W1490165506,60,Introduce youth quotas in parliaments to ensur...,630,False
4,W153963444,2,Prize-money for forest owners who retain habit...,989,False
...,...,...,...,...,...
15729,W60052349,83,Allow States to exempt farm trucks with a gros...,896,False
15730,W60052349,83,Allow States to exempt farm trucks with a gros...,896,False
15731,W60052349,83,Allow States to exempt farm trucks with a gros...,896,False
15732,W60052349,83,Allow States to exempt farm trucks with a gros...,896,False


In [63]:
merge = gold.drop(columns=['openalex_id', 'chunk_idx', 'text']).merge(gp.drop(columns=['openalex_id', 'chunk_idx', 'representative']), left_on=['policy'], right_on=['text'], how='left').drop(columns='text')
merge

,policy,impact_category,impact,direction,cluster_id
0,Introduce youth quotas in parliaments to ensur...,Justice,distributional,positive,630.0
1,Introduce youth quotas in parliaments to ensur...,Justice,distributional,positive,630.0
2,Provide exit options for young people to leave...,Justice,distributional,positive,540.0
3,Provide exit options for young people to leave...,Justice,distributional,positive,540.0
4,Prize-money for forest owners who retain habit...,PlanetaryBoundary,biosphere_integrity,positive,989.0
...,...,...,...,...,...
125614,National exemption from interstate commerce re...,Justice,procedural,positive,872.0
125615,National exemption from interstate commerce re...,Justice,procedural,positive,872.0
125616,National exemption from interstate commerce re...,Justice,procedural,positive,872.0
125617,National exemption from interstate commerce re...,Justice,procedural,positive,872.0


In [53]:
merge = merge.rename(columns={'impact': 'impact_dim'})

In [56]:
merge['impact_dim'] = merge['impact_dim'].apply(lambda x: x.replace('_', ' ').capitalize())

In [60]:
merge.merge(df, on=['cluster_id', 'impact_category', 'impact_dim'], how='left')

,policy,impact_category,impact_dim,direction,cluster_id,openalex_id,chunk_idx,impact_direction,chunk_id
0,Introduce youth quotas in parliaments to ensur...,Justice,Distributional,positive,630,W1576299163,0.0,positive,W1576299163_0
1,Introduce youth quotas in parliaments to ensur...,Justice,Distributional,positive,630,W2129700225,0.0,positive,W2129700225_0
2,Introduce youth quotas in parliaments to ensur...,Justice,Distributional,positive,630,W2198603358,3.0,positive,W2198603358_3
3,Introduce youth quotas in parliaments to ensur...,Justice,Distributional,positive,630,W2209627884,1.0,positive,W2209627884_1
4,Introduce youth quotas in parliaments to ensur...,Justice,Distributional,positive,630,W2560408720,2.0,positive,W2560408720_2
...,...,...,...,...,...,...,...,...,...
3550979,National exemption from interstate commerce re...,Justice,Procedural,positive,872,NaN,NaN,NaN,NaN
3550980,National exemption from interstate commerce re...,Justice,Procedural,positive,872,NaN,NaN,NaN,NaN
3550981,National exemption from interstate commerce re...,Justice,Procedural,positive,872,NaN,NaN,NaN,NaN
3550982,National exemption from interstate commerce re...,Justice,Procedural,positive,872,NaN,NaN,NaN,NaN
